# Bank Customer Churn Prediction

## Notebook 2: Data Preprocessing and Feature Engineering

This notebook prepares the dataset for machine learning.

Steps covered:
- Load the original dataset
- Check missing values and duplicates
- Remove unnecessary columns
- Create new features
- Separate features and target
- Split the data using stratification
- Encode categorical variables
- Scale numerical variables
- Save the processed data and preprocessing pipeline

In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

In [2]:
df = pd.read_csv("European_Bank.csv")

print("Shape:", df.shape)
df.head()

Shape: (10000, 14)


,Year,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,2025,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2025,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,2025,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,2025,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,2025,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [3]:
missing = df.isnull().sum()

print(missing[missing > 0])

Series([], dtype: int64)


In [4]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


In [5]:
print(df["Exited"].value_counts())
print()
print((df["Exited"].value_counts(normalize=True) * 100).round(2))

Exited
0    7963
1    2037
Name: count, dtype: int64

Exited
0    79.63
1    20.37
Name: proportion, dtype: float64


In [6]:
df = df.drop(columns=["CustomerId", "Surname"])

df.head()

,Year,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,2025,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2025,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,2025,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,2025,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,2025,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [7]:
df["BalanceSalaryRatio"] = (
    df["Balance"] / (df["EstimatedSalary"] + 1)
)

df["ProductDensity"] = (
    df["NumOfProducts"] / (df["Tenure"] + 1)
)

df["EngagementProductInteraction"] = (
    df["IsActiveMember"] * df["NumOfProducts"]
)

df["AgeTenureInteraction"] = (
    df["Age"] * df["Tenure"]
)

In [8]:
new_features = [
    "BalanceSalaryRatio",
    "ProductDensity",
    "EngagementProductInteraction",
    "AgeTenureInteraction"
]

df[new_features].describe().T

,count,mean,std,min,25%,50%,75%,max
BalanceSalaryRatio,10000.0,3.790150,100.055758,0.000000,0.000000,0.746998,1.514002,9770.883148
ProductDensity,10000.0,0.367173,0.337760,0.090909,0.166667,0.250000,0.500000,3.000000
EngagementProductInteraction,10000.0,0.791000,0.870169,0.000000,0.000000,1.000000,2.000000,4.000000
AgeTenureInteraction,10000.0,194.804000,127.939996,0.000000,90.000000,180.000000,280.000000,880.000000


In [9]:
df.head()

,Year,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,BalanceSalaryRatio,ProductDensity,EngagementProductInteraction,AgeTenureInteraction
0,2025,619,France,Female,42,2,0.00,1,1,1,101348.88,1,0.000000,0.333333,1,84
1,2025,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,0.744670,0.500000,1,41
2,2025,502,France,Female,42,8,159660.80,3,1,0,113931.57,1,1.401362,0.333333,0,336
3,2025,699,France,Female,39,1,0.00,2,0,0,93826.63,0,0.000000,1.000000,0,39
4,2025,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0,1.587035,0.333333,1,86


In [10]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Rows: 10000
Columns: 16


In [11]:
print("Missing values:")
print(df.isnull().sum().sum())

print("\nInfinite values:")
print(np.isinf(df.select_dtypes(include=np.number)).sum().sum())

Missing values:
0

Infinite values:
0


In [12]:
df = df.replace([np.inf, -np.inf], np.nan)

In [13]:
print(df.isnull().sum())

Year                            0
CreditScore                     0
Geography                       0
Gender                          0
Age                             0
Tenure                          0
Balance                         0
NumOfProducts                   0
HasCrCard                       0
IsActiveMember                  0
EstimatedSalary                 0
Exited                          0
BalanceSalaryRatio              0
ProductDensity                  0
EngagementProductInteraction    0
AgeTenureInteraction            0
dtype: int64


In [14]:
X = df.drop(columns=["Exited"])
y = df["Exited"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (10000, 15)
y shape: (10000,)


In [15]:
numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

Numerical features:
['Year', 'CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'BalanceSalaryRatio', 'ProductDensity', 'EngagementProductInteraction', 'AgeTenureInteraction']

Categorical features:
['Geography', 'Gender']


C:\Users\HP\AppData\Local\Temp\ipykernel_11244\1263442432.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(


In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data :", X_test.shape)

Training data: (8000, 15)
Testing data : (2000, 15)


In [17]:
print("Training set:")
print((y_train.value_counts(normalize=True) * 100).round(2))

print("\nTest set:")
print((y_test.value_counts(normalize=True) * 100).round(2))

Training set:
Exited
0    79.62
1    20.38
Name: proportion, dtype: float64

Test set:
Exited
0    79.65
1    20.35
Name: proportion, dtype: float64


In [18]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

In [19]:
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

In [20]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape :", X_test_processed.shape)

Processed training shape: (8000, 18)
Processed testing shape : (2000, 18)


In [21]:
feature_names = preprocessor.get_feature_names_out()

print("Number of final features:", len(feature_names))

print("\nFirst few features:")
print(feature_names[:20])

Number of final features: 18

First few features:
['num__Year' 'num__CreditScore' 'num__Age' 'num__Tenure' 'num__Balance'
 'num__NumOfProducts' 'num__HasCrCard' 'num__IsActiveMember'
 'num__EstimatedSalary' 'num__BalanceSalaryRatio' 'num__ProductDensity'
 'num__EngagementProductInteraction' 'num__AgeTenureInteraction'
 'cat__Geography_France' 'cat__Geography_Germany' 'cat__Geography_Spain'
 'cat__Gender_Female' 'cat__Gender_Male']


In [22]:
X_train_processed = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_test_processed = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)

X_train_processed.head()

,num__Year,num__CreditScore,num__Age,num__Tenure,num__Balance,num__NumOfProducts,num__HasCrCard,num__IsActiveMember,num__EstimatedSalary,num__BalanceSalaryRatio,num__ProductDensity,num__EngagementProductInteraction,num__AgeTenureInteraction,cat__Geography_France,cat__Geography_Germany,cat__Geography_Spain,cat__Gender_Female,cat__Gender_Male
2151,0.0,1.058568,1.715086,0.684723,-1.226059,-0.910256,0.641042,-1.030206,1.042084,-0.036843,-0.719828,-0.908608,1.580589,1.0,0.0,0.0,0.0,1.0
8392,0.0,0.913626,-0.659935,-0.696202,0.413288,-0.910256,0.641042,-1.030206,-0.623556,-0.022540,-0.346241,-0.908608,-0.770341,0.0,1.0,0.0,0.0,1.0
5006,0.0,1.079274,-0.184931,-1.731895,0.601687,0.808830,0.641042,0.970680,0.308128,-0.028152,4.883987,1.390574,-1.515190,0.0,1.0,0.0,1.0,0.0
4117,0.0,-0.929207,-0.184931,-0.005739,-1.226059,0.808830,0.641042,-1.030206,-0.290199,-0.036843,-0.097182,-0.908608,-0.079804,1.0,0.0,0.0,0.0,1.0
7182,0.0,0.427035,0.955079,0.339492,0.548318,0.808830,-1.559960,0.970680,0.135042,-0.027627,-0.239501,1.390574,0.765910,0.0,1.0,0.0,0.0,1.0


In [23]:
print("Training data shape:", X_train_processed.shape)
print("Testing data shape :", X_test_processed.shape)

print("\nMissing values in training data:",
      X_train_processed.isnull().sum().sum())

print("Missing values in testing data:",
      X_test_processed.isnull().sum().sum())

Training data shape: (8000, 18)
Testing data shape : (2000, 18)

Missing values in training data: 0
Missing values in testing data: 0


In [24]:
numeric_processed_columns = [
    col for col in X_train_processed.columns
    if col.startswith("num__")
]

X_train_processed[numeric_processed_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
num__Year,8000.0,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
num__CreditScore,8000.0,-5.728751e-16,1.000063,-3.113690,-0.691088,0.012915,0.685860,2.062809
num__Age,8000.0,2.486900e-16,1.000063,-1.989948,-0.659935,-0.184931,0.480075,5.040117
num__Tenure,8000.0,-9.814372e-17,1.000063,-1.731895,-0.696202,-0.005739,1.029954,1.720416
num__Balance,8000.0,5.773160e-17,1.000063,-1.226059,-1.226059,0.331855,0.822689,2.600500
num__NumOfProducts,8000.0,-9.503509e-17,1.000063,-0.910256,-0.910256,-0.910256,0.808830,4.247004
num__HasCrCard,8000.0,4.440892e-18,1.000063,-1.559960,-1.559960,0.641042,0.641042,0.641042
num__IsActiveMember,8000.0,4.263256e-17,1.000063,-1.030206,-1.030206,0.970680,0.970680,0.970680
num__EstimatedSalary,8000.0,-6.694645e-17,1.000063,-1.739344,-0.855358,-0.004951,0.851136,1.748805
num__BalanceSalaryRatio,8000.0,8.881784e-19,1.000063,-0.036843,-0.036843,-0.030125,-0.023299,87.514584


In [25]:
categorical_processed_columns = [
    column for column in X_train_processed.columns
    if column.startswith("cat__")
]

categorical_processed_columns

['cat__Geography_France',
 'cat__Geography_Germany',
 'cat__Geography_Spain',
 'cat__Gender_Female',
 'cat__Gender_Male']

In [26]:
print("X_train:", X_train_processed.shape)
print("y_train:", y_train.shape)

print("X_test:", X_test_processed.shape)
print("y_test:", y_test.shape)

X_train: (8000, 18)
y_train: (8000,)
X_test: (2000, 18)
y_test: (2000,)


In [27]:
X_train_processed.to_csv("X_train.csv",index=False)
X_test_processed.to_csv("X_test.csv",index=False)
y_train.to_csv("y_train.csv",index=False)
y_test.to_csv("y_test.csv",index=False)

print("Processed datasets saved.")

Processed datasets saved.


In [28]:
X_train.to_csv("X_train_raw.csv",index=False)
X_test.to_csv("X_test_raw.csv",index=False)

print("Raw train/test feature sets saved.")

Raw train/test feature sets saved.


In [29]:
joblib.dump(
    preprocessor,
    "preprocessor.pkl"
)

print("Preprocessor saved.")

Preprocessor saved.


In [30]:
joblib.dump(
    list(feature_names),
    "feature_names.pkl"
)

print("Feature names saved.")

Feature names saved.


In [31]:
print("Preprocessing completed successfully.\n")

print("Training samples :", len(X_train))
print("Testing samples  :", len(X_test))

print("Training features:", X_train_processed.shape[1])

print("\nTraining churn rate:",
      round(y_train.mean() * 100, 2), "%")

print("Testing churn rate:",
      round(y_test.mean() * 100, 2), "%")

Preprocessing completed successfully.

Training samples : 8000
Testing samples  : 2000
Training features: 18

Training churn rate: 20.38 %
Testing churn rate: 20.35 %


# Conclusion

The bank customer dataset has been prepared for machine learning.

The preprocessing stage included:
- Removing duplicate rows
- Removing `CustomerId` and `Surname`
- Creating four engineered features
- Separating predictors and target
- Performing a stratified 80:20 train-test split
- Handling missing numerical and categorical values
- One-hot encoding categorical variables
- Scaling numerical variables
- Saving the preprocessing pipeline and processed datasets

The data is now ready for model development.

The next notebook will train and compare:
- Logistic Regression
- Decision Tree
- Random Forest
- Gradient Boosting
- XGBoost

using cross-validation and multiple evaluation metrics.